In [ ]:
# Week 5: Inventory Health & Turnover Analysis

import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# 1. Load Data
data_path = '../data/'
inventory_df = pd.read_csv(os.path.join(data_path, 'engineered_inventory.csv'))
products_df = pd.read_csv(os.path.join(data_path, 'validated_products.csv'))
product_perf_df = pd.read_csv(os.path.join(data_path, 'processed/product_performance_summary.csv'))

# 2. Calculate Inventory Value & Metrics
inventory_health = inventory_df.merge(products_df[['product_id', 'unit_cost', 'unit_price', 'category']], on='product_id', how='left')
inventory_health = inventory_health.merge(product_perf_df[['product_id', 'total_units_sold', 'total_revenue', 'movement_classification']], on='product_id', how='left').fillna({'total_units_sold': 0, 'total_revenue': 0})

inventory_health['total_inventory_value'] = inventory_health['current_stock_level'] * inventory_health['unit_cost']

# 3. Overstock & Understock Risk Identification
def classify_stock_health(row):
    if row['current_stock_level'] <= row['reorder_point']:
        return 'Understocked / Risk'
    elif row['stock_status'] == 'Reorder Needed':
        return 'Understocked / Risk'
    elif row['inventory_turnover_ratio'] < 1.0 and row['current_stock_level'] > (row['reorder_point'] * 2):
        return 'Overstocked'
    else:
        return 'Healthy'

inventory_health['stock_health_status'] = inventory_health.apply(classify_stock_health, axis=1)

# 4. ABC / Pareto Classification
inventory_health = inventory_health.sort_values(by='total_inventory_value', ascending=False).reset_index(drop=True)
inventory_health['cum_value'] = inventory_health['total_inventory_value'].cumsum()
total_val = inventory_health['total_inventory_value'].sum()
inventory_health['cum_pct'] = (inventory_health['cum_value'] / total_val) * 100

def assign_abc(pct):
    if pct <= 70:
        return 'A'
    elif pct <= 90:
        return 'B'
    else:
        return 'C'

inventory_health['abc_classification'] = inventory_health['cum_pct'].apply(assign_abc)

# 5. Save Deliverable
output_path = '../data/processed/'
os.makedirs(output_path, exist_ok=True)
inventory_health.to_csv(os.path.join(output_path, 'inventory_health_summary.csv'), index=False)
print('Week 5 inventory pipeline completed successfully.')